# Sentiment Analysis

**Oasis Infobyte Data Analytics Internship — Level 1, Task 4**

This notebook classifies text into negative, neutral and positive sentiment using TF-IDF features and two machine-learning classifiers.


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from wordcloud import WordCloud

sns.set_theme(style="whitegrid")
np.random.seed(42)


## 2. Load the Dataset

The TweetEval sentiment dataset contains three sentiment classes: negative, neutral and positive.


In [ ]:
dataset = load_dataset("tweet_eval", "sentiment")

train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = ["negative", "neutral", "positive"]

train_df["label_name"] = train_df["label"].map(dict(enumerate(label_names)))
test_df["label_name"] = test_df["label"].map(dict(enumerate(label_names)))

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

display(train_df[["text", "label_name"]].head())


## 3. Inspect Class Distribution


In [ ]:
class_distribution = train_df["label_name"].value_counts().reindex(label_names)

display(class_distribution.to_frame("count"))

plt.figure(figsize=(8, 5))
sns.barplot(x=class_distribution.index, y=class_distribution.values)
plt.title("Sentiment Distribution — Training Data")
plt.xlabel("Sentiment")
plt.ylabel("Number of Tweets")
plt.tight_layout()
plt.show()


## 4. Text Preprocessing

The preprocessing pipeline performs the following operations:

1. Convert text to lowercase.
2. Remove URLs.
3. Remove punctuation.
4. Remove extra whitespace.

Stopword removal and tokenisation are handled by the TF-IDF vectorizer.


In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["text"].apply(preprocess_text)
test_df["clean_text"] = test_df["text"].apply(preprocess_text)

display(train_df[["text", "clean_text"]].head())


## 5. TF-IDF Feature Extraction

**TF-IDF** stands for Term Frequency-Inverse Document Frequency.

It represents each document as a numerical vector. A word receives a higher weight when it is important within a document but appears in relatively few documents across the complete corpus.

The vectorizer below also removes English stopwords and tokenises the cleaned text into features.


In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_train = tfidf.fit_transform(train_df["clean_text"])
X_test = tfidf.transform(test_df["clean_text"])

y_train = train_df["label"]
y_test = test_df["label"]

print("Training matrix:", X_train.shape)
print("Testing matrix:", X_test.shape)


## 6. Train/Test Split — 80/20

The internship checklist requires an 80/20 train-test split. Because TweetEval already provides official train and test partitions, this notebook combines those partitions and creates a reproducible stratified 80/20 split for the task.


In [ ]:
all_text = pd.concat(
    [train_df[["clean_text", "label"]], test_df[["clean_text", "label"]]],
    ignore_index=True
)

X_text_train, X_text_test, y_train, y_test = train_test_split(
    all_text["clean_text"],
    all_text["label"],
    test_size=0.20,
    random_state=42,
    stratify=all_text["label"]
)

print("Training samples:", len(X_text_train))
print("Testing samples:", len(X_text_test))

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)

X_train = tfidf.fit_transform(X_text_train)
X_test = tfidf.transform(X_text_test)


## 7. Model 1 — Multinomial Naive Bayes


In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

print(classification_report(
    y_test,
    nb_pred,
    target_names=label_names,
    digits=4
))


## 8. Model 2 — Logistic Regression


In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print(classification_report(
    y_test,
    lr_pred,
    target_names=label_names,
    digits=4
))


## 9. Compare Model Performance


In [ ]:
def model_metrics(y_true, y_pred, model_name):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Macro)": precision,
        "Recall (Macro)": recall,
        "F1 (Macro)": f1
    }

results = pd.DataFrame([
    model_metrics(y_test, nb_pred, "Multinomial Naive Bayes"),
    model_metrics(y_test, lr_pred, "Logistic Regression")
])

display(results.round(4))


## 10. Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, title in zip(
    axes,
    [nb_pred, lr_pred],
    ["Naive Bayes", "Logistic Regression"]
):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_names,
        yticklabels=label_names,
        ax=ax
    )
    ax.set_title(f"Confusion Matrix — {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## 11. WordCloud — Negative Sentiment


In [ ]:
negative_text = " ".join(
    all_text.loc[all_text["label"] == 0, "clean_text"]
)

negative_wc = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(negative_text)

plt.figure(figsize=(12, 6))
plt.imshow(negative_wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud — Negative Sentiment")
plt.show()


## 12. WordCloud — Neutral Sentiment


In [ ]:
neutral_text = " ".join(
    all_text.loc[all_text["label"] == 1, "clean_text"]
)

neutral_wc = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(neutral_text)

plt.figure(figsize=(12, 6))
plt.imshow(neutral_wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud — Neutral Sentiment")
plt.show()


## 13. WordCloud — Positive Sentiment


In [ ]:
positive_text = " ".join(
    all_text.loc[all_text["label"] == 2, "clean_text"]
)

positive_wc = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(positive_text)

plt.figure(figsize=(12, 6))
plt.imshow(positive_wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud — Positive Sentiment")
plt.show()


## 14. Error Analysis

The model with the higher macro F1-score is selected for error analysis.

Five incorrectly classified test examples are displayed below. Review them for patterns such as sarcasm, ambiguous language, negation, context dependence, slang and short messages.


In [ ]:
best_model_name = results.sort_values("F1 (Macro)", ascending=False).iloc[0]["Model"]
best_pred = lr_pred if best_model_name == "Logistic Regression" else nb_pred

error_frame = pd.DataFrame({
    "text": X_text_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(best_pred)
})

errors = error_frame[
    error_frame["actual"] != error_frame["predicted"]
].head(5)

errors["actual"] = errors["actual"].map(dict(enumerate(label_names)))
errors["predicted"] = errors["predicted"].map(dict(enumerate(label_names)))

print("Selected model:", best_model_name)
display(errors)


### Error Analysis Discussion

Misclassification is expected in short social-media text. Some errors can arise because sentiment depends on context, sarcasm, negation or informal expressions that are difficult to represent using word-level TF-IDF features.

Neutral and positive/negative examples can also share vocabulary, while very short messages may not contain enough discriminative information for a traditional linear classifier.


## 15. Conclusion

This project demonstrates an end-to-end three-class sentiment analysis workflow.

The final comparison table should be used to identify which of the two models achieved the stronger macro F1-score on the reproducible 20% test split.

A sentiment classifier of this type can be applied to customer feedback, product reviews, social-media monitoring, brand analysis and customer-support triage.

The model should be retrained and monitored when the language, domain or user population changes.


## 16. Save Evaluation Results


In [ ]:
results.to_csv("sentiment_model_comparison.csv", index=False)
errors.to_csv("sentiment_error_examples.csv", index=False)

print("Saved:")
print("- sentiment_model_comparison.csv")
print("- sentiment_error_examples.csv")
